In [2]:
!python --version


Python 3.11.13


In [4]:
!pip install tensorflow

In [5]:
import tensorflow as tf
print(tf.__version__)


2.18.0


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files

uploaded = files.upload()

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)



df = pd.read_csv('sovereign_default_dataset_1980_2022.csv - sovereign_default_dataset_1980_2022.csv')


df = df.sort_values(['Country', 'Year'])
features = [
    'Debt_GDP', 'ExtDebt_GDP', 'DebtServ_XGDP', 'RealGDP_growth', 'GDP_per_capita_USD',
    'Inflation', 'fiscal_balance', 'primary_balance', 'CurrentAccount_GDP', 'Reserves_months',
    'ExchangeRate_change', 'Trade_openness', 'US_FedFundsRate', 'World_GDP_growth',
    'Oil_price_index', 'VIX', 'ICRG_political', 'ElectionYear'
]

#  Prepare sequences (5 years sliding window)
sequence_length = 5
X_sequences, y_labels = [], []

for country in df['Country'].unique():
    country_df = df[df['Country'] == country]
    for i in range(len(country_df) - sequence_length):
        seq_X = country_df.iloc[i:i+sequence_length][features].values
        target_y = country_df.iloc[i+sequence_length]['Default']
        X_sequences.append(seq_X)
        y_labels.append(target_y)

X_sequences = np.array(X_sequences)
y_labels = np.array(y_labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# LSTM model
model = Sequential([
    LSTM(64, input_shape=(sequence_length, len(features)), return_sequences=False),
    Dropout(0.3),  # Slightly higher dropout to reduce overfitting
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping callback for efficiency
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=20,  # More epochs original 10
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Extract LSTM embeddings
# Rebuild the model with Functional API so we can access intermediate layers more reliably
from tensorflow.keras.layers import Input

input_layer = Input(shape=(sequence_length, len(features)))
x = LSTM(64, return_sequences=False, name='lstm_layer')(input_layer)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
output_layer = Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Re-train the functional model
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# ✅ Now we can safely extract the output from the LSTM layer
extractor = tf.keras.Model(inputs=model.input, outputs=model.get_layer('lstm_layer').output)
X_train_embed = extractor.predict(X_train)
X_test_embed = extractor.predict(X_test)

# Flatten for XGBoost
X_train_embed_flat = X_train_embed.reshape((X_train_embed.shape[0], -1))
X_test_embed_flat = X_test_embed.reshape((X_test_embed.shape[0], -1))

# XGBoost on embeddings
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=100,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()  # Handle imbalance
)

xgb_model.fit(X_train_embed_flat, y_train)

#  Evaluate
y_pred = xgb_model.predict(X_test_embed_flat)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Hybrid model Accuracy: {accuracy:.3f}")
print(f"Hybrid model F1 Score: {f1:.3f}")


Saving sovereign_default_dataset_1980_2022.csv - sovereign_default_dataset_1980_2022.csv to sovereign_default_dataset_1980_2022.csv - sovereign_default_dataset_1980_2022 (1).csv
Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.5186 - loss: 0.7084 - val_accuracy: 0.9603 - val_loss: 0.2276
Epoch 2/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9401 - loss: 0.2513 - val_accuracy: 0.9603 - val_loss: 0.1704
Epoch 3/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9401 - loss: 0.2306 - val_accuracy: 0.9603 - val_loss: 0.1700
Epoch 4/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9401 - loss: 0.2322 - val_accuracy: 0.9603 - val_loss: 0.1687
Epoch 5/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9401 - loss: 0.2423 - val_accuracy: 0.9603 - val_loss: 0.1690
Epoch 6/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9401 - loss: 0.2383 - val_accuracy: 0.9603 - val_loss: 0.1694
Epoch 7/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9401 - loss: 0.2350 - val_accuracy: 0.9603 - val_loss: 0.1692
Epoch 1/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8182 - loss: 0.4871 - val_accuracy: 0.9603 - val_loss: 0.176

/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [22:55:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
